In [1]:
from dfbr.utils.files import get_config, get_path
from dfbr.data.dataset import BikeDemandDataset, BikeOptTargetsDataset
from dfbr.models.station_targets import BikeStationTargets
from dfbr.models.cost_head import CostHead
from dfbr.models.mlp import MLP
from dfbr.eval.simulation import Sim, create_station_dict, create_event_df
from dfbr.training.train import evaluate
import datetime
import argparse
import shutil
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn as nn
import torch
import pyepo
import torch.nn.functional as F
import numpy as np

In [2]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Setup
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

#Read config
config = get_config("baseline.yaml")

#Create a dictionary of stations
station_dict = create_station_dict(config["paths"]["stations"], config["paths"]["station_dist_miles"], config["sim"]["start_inv_pct"])
#Sort by id to ensure alignment
station_ids = sorted(station_dict.keys())
#Get parameters for shapes of datasets and models
num_stations = len(station_dict)
capacities = [station_dict[sid].capacity for sid in station_ids]
max_cap = max(capacities)

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Load Datasets
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Create datasets
train_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["train_start_date"],
        end_date = config["data"]["train_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap
    )

training_stats = {'mean': train_ds.mean, 'std': train_ds.std, 'y_mean': train_ds.y_mean, 'y_std': train_ds.y_std}

test_ds = BikeDemandDataset(
        file = config["paths"]["input"],
        start_date = config["data"]["test_start_date"],
        end_date = config["data"]["test_end_date"],
        target_cols= [str(id) for id in station_ids],
        input_scale_cols= ['mean_temp', 'precip', 'max_gust'],
        input_no_scale_cols=['sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month'],
        capacities=capacities,
        max_cap=max_cap,
        is_train=False,
        scaling_factor=training_stats
    )

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Solve for optimal values with ground truth data
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
opt_model = BikeStationTargets(num_stations=num_stations, max_cap=max_cap, total_inventory=int(sum(capacities) * config["sim"]["start_inv_pct"]))
pyepo_train_ds = BikeOptTargetsDataset(opt_model, train_ds.X.numpy(), train_ds.c.view(-1, num_stations * (max_cap + 1)).numpy(), train_ds.y.numpy(), train_ds.dates)
pyepo_test_ds = BikeOptTargetsDataset(opt_model, test_ds.X.numpy(), test_ds.c.view(-1, num_stations * (max_cap + 1)).numpy(), test_ds.y.numpy(), test_ds.dates)

#Wrap Data Loaders
train_dl = DataLoader(pyepo_train_ds, batch_size=config["training"]["batch_size"], shuffle=True)
test_dl = DataLoader(pyepo_test_ds, batch_size=config["training"]["batch_size"], shuffle=False)

Set parameter Username
Set parameter LicenseID to value 2774727
Academic license - for non-commercial use only - expires 2027-02-03
Optimizing for optDataset...


100%|██████████| 519/519 [00:18<00:00, 28.81it/s]

Optimizing for optDataset...



100%|██████████| 365/365 [00:12<00:00, 30.00it/s]


In [3]:
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Instantiate Models and loss functions
#-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

#Create MLP
input_size = len(train_ds[0][0])
output_size = num_stations
pred_model = MLP(input_size, output_size, config["model"]["hidden_layers"])

#Create cost head
cost_head = CostHead(capacities=capacities, max_cap=max_cap)
full_model = nn.Sequential(pred_model, cost_head)

#Create optimizer
optimizer = Adam(full_model.parameters(), lr=config["training"]["learning_rate"])
spo = pyepo.func.SPOPlus(opt_model, processes=1)
mse = nn.MSELoss()

Num of cores: 1


In [ ]:
pred_model.eval()
total_samples = 0
dates = []
total_mse = []
true_demand = []
true_targets = []
true_obj = []
pred_demand = []
pred_targets = []
pred_obj = []
real_obj = []

with torch.no_grad():  
    for x, c, w, z, y, date in train_dl:
        #Record the ground truth solutions
        dates.extend(date)
        true_demand.append(y)
        true_obj.append(z)
        w = w.view(-1, cost_head.num_stations, cost_head.max_cap +1)
        true_targets.append(torch.argmax(w, axis = 2))

        #Get predictions and predicted cost function
        yp = pred_model(x)
        cp = cost_head(yp)
        #Record predictions
        pred_demand.append(yp)

        #Comute mse 
        batch_mse = F.mse_loss(yp, y)
        total_mse.append(batch_mse.item())
        total_samples += x.shape[0]

        #Compute total cost
        for i in range(x.shape[0]):
            opt_model.setObj(cp[i])
            wp, zp = opt_model.solve()
            real_cost = np.dot(wp,  c[i])
            #Get targets 
            target = torch.tensor(wp).view(cost_head.num_stations, cost_head.max_cap +1)
            #Record the predictions 
            pred_obj.append(zp)
            pred_targets.append(torch.argmax(target, axis = 1).tolist())
            real_obj.append(real_cost)

#Reshape into dataframe
df = pd.DataFrame({
    'split' : 'train',
    'date' : dates,
    'true_demand': torch.cat(true_demand, axis=0).tolist(),
    'true_targets': torch.cat(true_targets, axis=0).tolist(),
    'true_obj': torch.cat(true_obj, axis=0).squeeze().tolist(), 
    'pred_demand': torch.cat(pred_demand, axis=0).tolist(),
    'pred_targets': pred_targets,
    'pred_obj': pred_obj, 
    'real_obj': real_obj, 
})

yp: torch.Size([32, 60]) dummmy: torch.Size([32, 60])
yp: torch.Size([32, 60]) dummmy: torch.Size([32, 60])


KeyboardInterrupt: 

In [ ]:
print(df.sort_values('date').reset_index(drop=True))

     split        date                                        true_demand  \
0    train  2023-01-01  [0.0, 0.0, 1.0, -2.0, -1.0, -3.0, 5.0, 0.0, 0....   
1    train  2023-01-02  [0.0, 0.0, 1.0, -1.0, 0.0, -1.0, 0.0, 1.0, 0.0...   
2    train  2023-01-03  [0.0, -1.0, 0.0, 1.0, 0.0, -1.0, 0.0, -1.0, 0....   
3    train  2023-01-04  [0.0, 0.0, 0.0, 0.0, 0.0, -2.0, -2.0, -1.0, 0....   
4    train  2023-01-05  [0.0, -2.0, 0.0, 1.0, 4.0, 2.0, 1.0, -1.0, -1....   
..     ...         ...                                                ...   
726  train  2024-12-27  [1.0, -2.0, 0.0, -1.0, 2.0, 0.0, -1.0, 0.0, 0....   
727  train  2024-12-28  [0.0, 1.0, -2.0, -1.0, -2.0, -1.0, 3.0, 2.0, 0...   
728  train  2024-12-29  [0.0, -1.0, 0.0, 2.0, -2.0, 0.0, -1.0, 2.0, 0....   
729  train  2024-12-30  [-5.0, -3.0, 0.0, -1.0, 0.0, -1.0, 0.0, -2.0, ...   
730  train  2024-12-31  [-2.0, 1.0, 0.0, 2.0, 0.0, 0.0, 1.0, 1.0, 1.0,...   

                                          true_targets  true_obj  \
0    [0

: 

: 